# Phase 3: our encoder, a pixel MAE trained from scratch (Kaggle)

Pretrains a masked autoencoder on the robot's raw 160x160 frames from both cameras plus the joint stream, with no pretrained vision model. Then trains the same frozen probe as the DINOv2 arm on it, and prints the offline error table for the comparison.

**Before running:** Add Input → Your Work → `so101-extract`. Accelerator: GPU T4 x2. Internet: On.

**Time:** pretraining is the long part, roughly 3-5 h for 60 epochs on grasp_2. Sessions cap at 9 h; if it stops early, Save Version again with `RESUME = True` and it continues from the last epoch (attach this notebook's own previous output as a second input for that).

**After running:** Save Version → `/kaggle/working/outputs/pixel_mae` and `outputs/pixel_probe`.

In [ ]:
GIT_REPO = "https://github.com/yashica-patodia/so101-imitation-learning.git"
LEVEL = "A"
TARGET = "state"
EPOCHS_MAE = 60
EPOCHS_PROBE = 20
BATCH = 32
RESUME = False         # True to continue a pretraining run from a previous version's outputs/pixel_mae/last.pt
USE_ONLY = None        # e.g. ["grasp_2"]

In [ ]:
import os, glob, shutil
os.chdir("/kaggle/working")
!rm -rf /kaggle/working/repo && git clone -q {GIT_REPO} /kaggle/working/repo
!pip -q install pytest 2>&1 | tail -1
os.chdir("/kaggle/working/repo")
!python -m pytest tests -q 2>&1 | tail -1
found = sorted(glob.glob("/kaggle/input/**/meta.json", recursive=True))
found = [f for f in found if glob.glob(os.path.dirname(f) + "/episode_*.npz")]
FEATURE_DIRS = [os.path.dirname(f) for f in found if USE_ONLY is None or os.path.basename(os.path.dirname(f)) in USE_ONLY]
if not FEATURE_DIRS:
    print(os.popen("ls -R /kaggle/input | head -40").read()); raise SystemExit("no extracted features under /kaggle/input")
for d in FEATURE_DIRS: print(d, len(glob.glob(d + "/episode_*.npz")), "episodes")
FEATS = " ".join(FEATURE_DIRS)
OUT = "/kaggle/working/outputs"; os.makedirs(OUT, exist_ok=True)
if RESUME:
    prev = sorted(glob.glob("/kaggle/input/**/outputs/pixel_mae/last.pt", recursive=True))
    assert prev, "RESUME=True but no previous outputs/pixel_mae/last.pt among the inputs"
    shutil.copytree(os.path.dirname(prev[-1]), f"{OUT}/pixel_mae", dirs_exist_ok=True); print("resuming from", prev[-1])
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!free -g | head -2

## 1. Pixel MAE pretraining

In [ ]:
resume_flag = "--resume" if RESUME else ""
!python -u -m nano_vla.train.train_mae --kind pixel --features {FEATS} --out {OUT}/pixel_mae \
    --level {LEVEL} --epochs {EPOCHS_MAE} --batch {BATCH} {resume_flag} 2>&1 | grep --line-buffered -v Warning

## 2. What it learned: masked input / reconstruction / original

In [ ]:
import sys, torch, numpy as np, matplotlib.pyplot as plt
sys.path.insert(0, "/kaggle/working/repo")
from nano_vla.models.pixel_mae import PixelMAE
from nano_vla.data.windows import EpisodeSet, WindowDataset, split_episodes
ck = torch.load(f"{OUT}/pixel_mae/best.pt", map_location="cpu"); cfg = ck["cfg"]
mae = PixelMAE(cfg["n_cams"], cfg["n_patches"], d=cfg["d"], depth=cfg["depth"], heads=cfg["heads"], dec_d=cfg["dec_d"], dec_depth=cfg["dec_depth"]); mae.load_state_dict(ck["model"]); mae.eval()
stats = dict(np.load(f"{OUT}/pixel_mae/stats.npz"))
n = len(glob.glob(FEATURE_DIRS[0] + "/episode_*.npz")); _, va = split_episodes(n, 0.1)
ds = WindowDataset(EpisodeSet(FEATURE_DIRS[0], va[:3], LEVEL, "pixel"), stats, 1.0)
b = ds[len(ds)//2]
torch.manual_seed(0)
mi, rec, orig = mae.reconstruct(b["frm"][None], b["state"][None])
fig, ax = plt.subplots(3, cfg["n_cams"] * 2, figsize=(4 * cfg["n_cams"] * 2, 12))
for c in range(cfg["n_cams"]):
    for k, t in enumerate([0, 4]):
        col = c * 2 + k
        for r, (name, im) in enumerate([("masked input", mi), ("reconstruction", rec), ("original", orig)]):
            ax[r][col].imshow(im[0, c, t].permute(1, 2, 0).numpy()); ax[r][col].set_title(f"{name} cam{c} t={t}"); ax[r][col].axis("off")
plt.tight_layout(); plt.show()

## 3. Frozen probe on the pixel MAE (same head as the DINOv2 arm)

In [ ]:
!python -u -m nano_vla.train.train_probe --mae {OUT}/pixel_mae/best.pt --features {FEATS} --out {OUT}/pixel_probe \
    --level {LEVEL} --target {TARGET} --epochs {EPOCHS_PROBE} --batch {BATCH} 2>&1 | grep --line-buffered -v Warning

In [ ]:
import json
print(open(f"{OUT}/pixel_probe/eval.txt").read())
mrec = [json.loads(l) for l in open(f"{OUT}/pixel_mae/log.jsonl")]
plt.figure(figsize=(6,3)); plt.plot([r["train"]["masked"] for r in mrec], label="train masked"); plt.plot([r["val"]["masked"] for r in mrec], label="val masked"); plt.legend(); plt.xlabel("epoch"); plt.title("pixel MAE loss"); plt.show()
!du -sh {OUT}/*